<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_26_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DB_USER = "prog_academy_da_157g_user"
DB_PASSWORD = "XTE8IHOEtQZ5GasTkaOjcGMKYTha6LeO"
DB_HOST = "dpg-d8t957gg4nts73da5bg0-a.frankfurt-postgres.render.com"
DB_PORT = "5432"
DB_NAME = "prog_academy_da_157g"


url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"


from sqlalchemy import create_engine

engine = create_engine(url)

import pandas as pd

## Task 1. Кумулятивна виручка користувача
Як накопичувалася сума замовлень у кожного користувача з часом?

In [6]:
query = """
SELECT  user_uuid,
        order_time,
        SUM(total_uah) OVER (
          PARTITION BY user_uuid
          ORDER BY order_time
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,order_time,sum
0,USR0000,2024-01-03,1857.51
1,USR0000,2024-01-31,3340.70
2,USR0000,2024-02-22,5223.80
3,USR0000,2024-03-23,5423.69


## Task 2. Інтервал між покупками
Скільки днів минуло з попереднього замовлення користувача?

In [8]:
query = """
SELECT  user_uuid,
        order_time,
        total_uah,
        order_time - LAG(order_time) OVER (
          PARTITION BY user_uuid
          ORDER BY order_time
        ) AS days_diff
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,order_time,total_uah,days_diff
0,USR0000,2024-01-03,1857.51,NaN
1,USR0000,2024-01-31,1483.19,28.0
2,USR0000,2024-02-22,1883.10,22.0
3,USR0000,2024-03-23,199.89,30.0


## Task 3. Попередня сума покупки
Якою була сума попереднього замовлення користувача?

In [10]:
query = """
SELECT  user_uuid,
        order_time,
        total_uah,
        LAG(total_uah) OVER (
          PARTITION BY user_uuid
          ORDER BY order_time
        )
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,order_time,total_uah,lag
0,USR0000,2024-01-03,1857.51,NaN
1,USR0000,2024-01-31,1483.19,1857.51
2,USR0000,2024-02-22,1883.10,1483.19
3,USR0000,2024-03-23,199.89,1883.10


## Task 4. Максимальне замовлення користувача
Яке найбільше замовлення було у кожного користувача?

In [11]:
query = """
SELECT  user_uuid,
        order_time,
        total_uah,
        MAX(total_uah) OVER (
          PARTITION BY user_uuid
        ) as max_order
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,order_time,total_uah,max_order
0,USR0000,2024-03-23,199.89,1883.1
1,USR0000,2024-01-03,1857.51,1883.1
2,USR0000,2024-02-22,1883.10,1883.1
3,USR0000,2024-01-31,1483.19,1883.1


## Task 5. Накопичувальне середнє чека
Як змінюється середній чек користувача від замовлення до замовлення?

In [14]:
query = """
SELECT  user_uuid,
        total_uah,
        order_time,
        AVG(total_uah) OVER (
          PARTITION BY user_uuid
          ORDER BY order_time
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,total_uah,order_time,avg
0,USR0000,1857.51,2024-01-03,1857.510000
1,USR0000,1483.19,2024-01-31,1670.350000
2,USR0000,1883.10,2024-02-22,1741.266667
3,USR0000,199.89,2024-03-23,1355.922500


In [17]:
query = """
SELECT  user_uuid,
        total_uah,
        order_time,
        AVG(total_uah) OVER (
          PARTITION BY user_uuid
          ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,total_uah,order_time,avg
0,USR0000,199.89,2024-03-23,1355.9225
1,USR0000,1857.51,2024-01-03,1355.9225
2,USR0000,1883.10,2024-02-22,1355.9225
3,USR0000,1483.19,2024-01-31,1355.9225


In [18]:
df[df.user_uuid == 'USR0000'].total_uah.mean()

np.float64(1355.9225000000001)

## Task 6. Останнє замовлення користувача
Яка сума останнього замовлення користувача?

In [19]:
query = """
SELECT  user_uuid,
        total_uah,
        order_time,
        LAST_VALUE(total_uah) OVER (
          PARTITION BY user_uuid
          ORDER BY order_time
          ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )
FROM orders_log
"""

df = pd.read_sql(query, engine)
df[df.user_uuid == 'USR0000']

,user_uuid,total_uah,order_time,last_value
0,USR0000,1857.51,2024-01-03,199.89
1,USR0000,1483.19,2024-01-31,199.89
2,USR0000,1883.10,2024-02-22,199.89
3,USR0000,199.89,2024-03-23,199.89
